# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end example for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is defined by the Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and record set definitions from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant metadata and dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset overview
metadata_json = dataset.metadata.to_json()  # This is a dict-like Python object
print("Name:", metadata_json['name'])
print("Description:", metadata_json['description'])


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

A **record set** corresponds to a structured table in the schema.

Let's list all record sets and, for each, print its `@id`, name, and fields with their `@id` and name.

In [ ]:
# List all record sets with IDs and their fields
record_sets = dataset.metadata.record_sets
if not record_sets:
    print('No record sets found in the Croissant schema.')
else:
    for rset in record_sets:
        print(f"Record Set: @id={rset.id}")
        print(f"          name={rset.name if hasattr(rset, 'name') else ''}")
        print("Fields:")
        for field in rset.fields:
            print(f"    @id={field.id}, name={field.name if hasattr(field, 'name') else ''}, dataType={field.data_type if hasattr(field, 'data_type') else ''}")
        print('-' * 60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

*Use the record set and field `@id`s from the overview above. Replace values below with actual IDs as needed.*

In [ ]:
# Choose one or more record sets to extract. Collect all available record set @ids into a list for demonstration.
record_set_ids = [rset.id for rset in dataset.metadata.record_sets]
dataframes = {}

# Load each record set into a DataFrame
for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}:", df.columns.tolist())
    if not df.empty:
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field and demonstrate basic filtering, normalization, and grouping.

*For demonstration, we'll use the first available record set and one of its numeric fields. Replace these with your actual field `@id`s as needed.*

In [ ]:
# Use first record set and try to pick a numeric field for analysis
import numpy as np
record_set_id = record_set_ids[0] if record_set_ids else None

if record_set_id is not None:
    df = dataframes[record_set_id]
    # Try to auto-detect a numeric field
    candidate_numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not candidate_numeric_fields:
        # Try to coerce columns to numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        candidate_numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()

    if not candidate_numeric_fields:
        print("No numeric columns available for analysis.")
    else:
        numeric_field = candidate_numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")

        # Remove nulls for numeric analysis
        numeric_data = df[numeric_field].dropna()
        if numeric_data.empty:
            print(f"No non-null data for field '{numeric_field}'")
        else:
            # Set a threshold for filtering, for demo pick mean
            threshold = numeric_data.mean() if numeric_data.mean() > 0 else 10
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold:.2f} (first 5 rows):")
            display(filtered_df.head())

            # Normalize
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() > 0 else 1)
            print(f"Normalized '{numeric_field}' (first 5 rows):")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Group by a categorical field if present
            candidate_group_fields = df.select_dtypes(include=[object]).columns.tolist()
            group_field = candidate_group_fields[0] if candidate_group_fields else None
            if group_field:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"Grouped mean for '{numeric_field}' by '{group_field}' (first 5 rows):")
                display(grouped_df.head())
            else:
                print('No categorical field found for grouping.')
else:
    print('No record sets available.')

## 5. Visualization
Visualize distributions or relationships for selected fields.

*Example: Histogram of a numeric field and boxplot grouped by a categorical field, if available.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None and 'numeric_field' in locals():
    df = dataframes[record_set_id]
    if numeric_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.show()

        # If grouping field present, show boxplot
        if 'group_field' in locals() and group_field in df.columns:
            plt.figure(figsize=(10, 4))
            sns.boxplot(data=df, x=group_field, y=numeric_field)
            plt.title(f"{numeric_field} by {group_field}")
            plt.xticks(rotation=30)
            plt.show()


## 6. Conclusion
In this notebook, we've loaded, explored, and visualized the FAIR^2 dataset using the `mlcroissant` library. We've demonstrated how to:

- Access dataset metadata and structure via `@id`s.
- List, load, and interact with record sets and their fields.
- Perform basic filtering, normalization, and aggregation.
- Visualize numeric distributions and group differences.

Customize the record set or field selection as needed for deeper exploration of your specific research or analysis questions.